# 01 - Genre Classification
**Spotify Data Mining | CISC 4631 | Group 3**

**Research Q:** can audio features predict genre?

**Approach:** LR (linear baseline) vs RF (bagging) vs XGBoost (boosting). compare accuracy, F1, feature importance.

> **Prereq:** run `00_data_setup.ipynb` first -> `df_genre_balanced.csv` on Drive.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# imports + shared constants (copy from Nb 00)
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay, confusion_matrix
import seaborn as sns

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

SEED            = 42
DRIVE_DATA_PATH = '/content/drive/MyDrive/data-mining-spotify-team3/cleanedData'
MODEL_PATH      = '/content/drive/MyDrive/data-mining-spotify-team3/models'
FIG_DIR         = '/content/drive/MyDrive/data-mining-spotify-team3/figures'
os.makedirs(FIG_DIR, exist_ok=True)
ALL_FEATURES = [
    'danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
    'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'key', 'mode'
]
np.random.seed(SEED)

## 1. Load Data

In [ ]:
# load balanced genre dataset from Drive (written by Nb 00)
df = pd.read_csv(os.path.join(DRIVE_DATA_PATH, 'df_genre_balanced.csv'))
print(f'Loaded: {df.shape}')
print('\nGenre counts:')
print(df['genre'].value_counts())
df.head(10)

## 2. Train / Eval / Test Split

60/20/20 stratified. Train: fit + 10-fold CV. Eval: compare/tune. Test: touched once at end for final numbers (prevents selecting models on test perf).

In [ ]:
# 2-step stratified split: first 80/20, then 75/25 of the 80 -> 60/20/20
X = df[ALL_FEATURES]
y = df['genre']

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

X_train, X_eval, y_train, y_eval = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=SEED, stratify=y_trainval
)

print(f'Train: {X_train.shape[0]:,} ({X_train.shape[0] / len(X):.0%})')
print(f'Eval:  {X_eval.shape[0]:,} ({X_eval.shape[0] / len(X):.0%})')
print(f'Test:  {X_test.shape[0]:,} ({X_test.shape[0] / len(X):.0%})')

print('\nClass distribution (train):')
print(y_train.value_counts())
print('\nClass distribution (eval):')
print(y_eval.value_counts())
print('\nClass distribution (test):')
print(y_test.value_counts())

## 3. Feature Ranking (MI)

Rank 12 audio features by mutual information with genre. MI > Pearson since it captures non-linear. Fit on train only (no eval/test leak).

In [ ]:
# fit MI scorer on train only
mi_scorer = SelectKBest(score_func=mutual_info_classif, k='all')
mi_scorer.fit(X_train, y_train)

mi_scores = pd.DataFrame({
    'Feature': ALL_FEATURES,
    'MI Score': mi_scorer.scores_
}).sort_values('MI Score', ascending=False).reset_index(drop=True)

print('Mutual information ranking:')
print(mi_scores.to_string(index=False))

# plot
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(mi_scores['Feature'][::-1], mi_scores['MI Score'][::-1], color='steelblue')
ax.set_xlabel('Mutual Information Score')
ax.set_title('Feature Ranking by Mutual Information with Genre (train only)')
plt.tight_layout()
plt.show()

# keep all 12 - MI is univariate, trees find interaction value in low-MI feats
# (tested K=10, lost ~1pp. XGBoost gives `mode` ~10% importance despite MI ~0.02)
K = 12
SELECTED_FEATURES = mi_scores['Feature'].head(K).tolist()
print(f'\nKeeping all {K} features (MI used as ranking diagnostic, not as filter):')
print(SELECTED_FEATURES)

## 4. Models

For each: fit on train, 10-fold CV on train for stability, eval set for tuning, hold test for final §5 numbers.

### 4.1 Logistic Regression (Baseline)

In [ ]:
# LR: needs scaling, use Pipeline. fit/CV/eval, store test preds for §5
lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, random_state=SEED))
])

lr_model.fit(X_train[SELECTED_FEATURES], y_train)

cv_lr = cross_val_score(
    lr_model, X_train[SELECTED_FEATURES], y_train, cv=10, scoring='accuracy'
)
print("CV Accuracy (train, 10-fold): %0.2f (+/- %0.2f)" % (cv_lr.mean(), cv_lr.std() * 2))

y_pred_lr_eval = lr_model.predict(X_eval[SELECTED_FEATURES])
print('\n--- EVAL SET ---')
print('Accuracy:', accuracy_score(y_eval, y_pred_lr_eval))
print(classification_report(y_eval, y_pred_lr_eval))

# final test pred (stored, not tuned against)
y_pred_lr = lr_model.predict(X_test[SELECTED_FEATURES])

### 4.2 Random Forest

In [ ]:
# RF: 100 trees, depth 15. trees are scale-invariant (no pipeline)
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=SEED,
    n_jobs=-1
)

rf_model.fit(X_train[SELECTED_FEATURES], y_train)

cv_rf = cross_val_score(
    rf_model, X_train[SELECTED_FEATURES], y_train, cv=10, scoring='accuracy'
)
print("CV Accuracy (train, 10-fold): %0.2f (+/- %0.2f)" % (cv_rf.mean(), cv_rf.std() * 2))

y_pred_rf_eval = rf_model.predict(X_eval[SELECTED_FEATURES])
print('\n--- EVAL SET ---')
print('Accuracy:', accuracy_score(y_eval, y_pred_rf_eval))
print(classification_report(y_eval, y_pred_rf_eval))

y_pred_rf = rf_model.predict(X_test[SELECTED_FEATURES])

### 4.3 XGBoost

In [ ]:
# XGBoost: needs int-encoded labels. tuned config: 500 trees, depth=10, lr=0.03, subsample 0.9
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_eval_enc  = le.transform(y_eval)
y_test_enc  = le.transform(y_test)

xgb_model = XGBClassifier(
    objective='multi:softmax',
    num_class=len(le.classes_),
    n_estimators=500,
    max_depth=10,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    eval_metric='mlogloss',
    random_state=SEED,
    n_jobs=-1,
)

xgb_model.fit(X_train[SELECTED_FEATURES], y_train_enc)

cv_xgb = cross_val_score(
    xgb_model, X_train[SELECTED_FEATURES], y_train_enc, cv=10, scoring='accuracy'
)
print("CV Accuracy (train, 10-fold): %0.2f (+/- %0.2f)" % (cv_xgb.mean(), cv_xgb.std() * 2))

# inverse_transform to get string labels back for classification_report
y_pred_xgb_eval = le.inverse_transform(xgb_model.predict(X_eval[SELECTED_FEATURES]))
print('\n--- EVAL SET ---')
print('Accuracy:', accuracy_score(y_eval, y_pred_xgb_eval))
print(classification_report(y_eval, y_pred_xgb_eval))

y_pred_xgb = le.inverse_transform(xgb_model.predict(X_test[SELECTED_FEATURES]))

## 5. Results (Test Set)

Final numbers - test touched once, no tuning against it.

### 5.1 Model Comparison (Test Set)

In [ ]:
# 3-way test-set comparison: acc / macro F1 / weighted F1
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ],
    'Macro F1': [
        f1_score(y_test, y_pred_lr, average='macro'),
        f1_score(y_test, y_pred_rf, average='macro'),
        f1_score(y_test, y_pred_xgb, average='macro')
    ],
    'Weighted F1': [
        f1_score(y_test, y_pred_lr, average='weighted'),
        f1_score(y_test, y_pred_rf, average='weighted'),
        f1_score(y_test, y_pred_xgb, average='weighted')
    ]
})

print(results.to_string(index=False))

In [ ]:
# bar chart: 3 models x 3 metrics
results_pct = results.copy()
results_pct[['Accuracy', 'Macro F1', 'Weighted F1']] *= 100

ax = results_pct.plot(x='Model', y=['Accuracy', 'Macro F1', 'Weighted F1'], kind='bar', figsize=(9, 5))
plt.title('Model Performance Comparison')
plt.ylabel('Score (%)')
plt.xticks(rotation=15, ha='right')
plt.ylim(0, 100)
# reference line for random baseline (14.3% = 1/7)
plt.axhline(100/7, color='red', linestyle='--', linewidth=1.2, label=f'Random baseline ({100/7:.1f}%)')
plt.legend(loc='upper left', fontsize=9)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'nb01_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Feature Importance — Random Forest vs XGBoost

In [ ]:
# side-by-side RF vs XGBoost feature importance
fi_df = pd.DataFrame({
    'Feature': SELECTED_FEATURES,
    'Random Forest': rf_model.feature_importances_,
    'XGBoost':       xgb_model.feature_importances_,
}).sort_values('Random Forest', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(fi_df))
width = 0.4
ax.bar(x - width/2, fi_df['Random Forest'], width, label='Random Forest', color='#2196F3', edgecolor='white')
ax.bar(x + width/2, fi_df['XGBoost'],       width, label='XGBoost',       color='#4CAF50', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(fi_df['Feature'], rotation=45, ha='right')
ax.set_ylabel('Importance')
ax.set_title('Feature Importance - Random Forest vs XGBoost')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'nb01_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

print(fi_df.to_string(index=False))

### 5.3 Confusion Matrix (XGBoost, Test Set)

In [ ]:
# XGBoost confusion matrix - row-normalized (each row = true genre, shows % of those songs per predicted class)
cm = confusion_matrix(y_test, y_pred_xgb, labels=le.classes_, normalize='true')

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(
    cm, annot=True, fmt='.0%', cmap='Blues',
    xticklabels=le.classes_, yticklabels=le.classes_,
    cbar_kws={'label': 'Fraction of true genre'}, ax=ax,
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('XGBoost Confusion Matrix (row-normalized)')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'nb01_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary

3 classifiers on 12 audio features: LR (linear) vs RF (bagging) vs XGBoost (boosting). XGBoost wins -> genre boundaries non-linear, boosting captures interactions LR/RF miss.

**Pipeline:**
1. 7 genres after consolidation, 6,895 per genre = 48,265 balanced.
2. 60/20/20 stratified split.
3. MI ranking as diagnostic (not filter). tested K=10 -> lost ~1pp -> kept all 12.
4. Fit on train, tune on eval, report on test.

**Baseline:** random = 14.3% (7-class). XGBoost ~52% = ~3.6x baseline.

**Pickle handoff:** XGBoost model + LabelEncoder + feature list saved to Drive for Nb 03's hybrid recommender.

---
## 7. Save Trained Model

Export XGBoost + metadata to Drive for Nb 03.

In [ ]:
# save model + label encoder + feature list for Nb 03 to load
os.makedirs(MODEL_PATH, exist_ok=True)

artifacts = {
    'genre_xgb_model.pkl':     xgb_model,
    'genre_label_encoder.pkl': le,
    'genre_feature_list.pkl':  SELECTED_FEATURES,
}

for fname, obj in artifacts.items():
    with open(os.path.join(MODEL_PATH, fname), 'wb') as f:
        pickle.dump(obj, f)
    print(f'  saved: {os.path.join(MODEL_PATH, fname)}')